|<h2>Course:</h2>|<h1><a href="https://derivingsystems.com/course.html" target="_blank">Build your own vLLM: inference engines from the memory system up</a></h1>|
|-|:-:|
|<h2>Part 1:</h2>|<h1>The Naive Loop<h1>|
|<h2>Section:</h2>|<h1>The arithmetic<h1>|
|<h2>Lecture:</h2>|<h1><b>CodeChallenge: size a deployment on paper<b></h1>|

<br>

<h5><b>Course repo:</b> <a href="https://github.com/Venugopalan2610/vllm-from-scratch" target="_blank">github.com/Venugopalan2610/vllm-from-scratch</a></h5>
<h5><b>The derivations:</b> <a href="https://derivingsystems.com" target="_blank">derivingsystems.com</a></h5>
<i>The notebooks build the intuition. The ladder in app/ makes you build the thing.</i>

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

import matplotlib_inline.backend_inline
matplotlib_inline.backend_inline.set_matplotlib_formats('svg')

You must size a deployment. You have one model and three cards. Somebody
wants two numbers. How many users at once, and how fast?

Everything here is arithmetic. You need no GPU, no measurement, and no slow
code. That is the point. You can answer this before you buy the hardware.

In [ ]:
### run this cell: three real cards, from their spec sheets

cards = {
  #                bandwidth B/s   bf16 FLOP/s     VRAM bytes
  'RTX 4090':     (1008e9,         165e12,         24e9),
  'A100 80GB':    (2039e9,         312e12,         80e9),
  'H100 SXM':     (3350e9,         989e12,         80e9),
}

# and one model: Llama-3-8B
params, layers, kv_heads, head_dim = 8e9, 32, 8, 128

# Exercise 1: the ridge point of each card

FLOP per byte. This is the batch size at which a card stops waiting for
memory.

In [ ]:
for name,(bandwidth, flops, vram) in cards.items():
  ridge = flops / bandwidth
  print(f'{name:12} {ridge:6.0f} FLOP/byte')

# Exercise 2: the speed limit at batch 1

One user runs alone on the machine. A token costs exactly one read of the
weights. This ceiling is a division, and no kernel beats it.

In [ ]:
dtype_bytes = 2
weight_bytes = params * dtype_bytes

for name,(bandwidth, flops, vram) in cards.items():
  read_ms  = 1000 * weight_bytes / bandwidth
  tokens_per_second = 1000 / read_ms
  print(f'{name:12} {read_ms:6.1f} ms/token   {tokens_per_second:6.0f} tok/s ceiling at batch 1')

# Exercise 3: the batch that memory permits

Subtract the weights. Divide the remainder by the KV cost of one sequence at
4096 tokens. Then compare your answer with Exercise 1.

In [ ]:
per_token = 2 * layers * kv_heads * head_dim * dtype_bytes
CONTEXT_LEN = 4096

print(f'{"card":12} {"KV room":>9} {"fits":>6} {"ridge":>7} {"verdict"}')
for name,(bandwidth, flops, vram) in cards.items():
  kv_room  = vram - weight_bytes
  num_fit  = kv_room / (CONTEXT_LEN * per_token)
  ridge = flops / bandwidth
  verdict = 'memory-bound' if num_fit < ridge else 'can reach the ridge'
  print(f'{name:12} {kv_room/1e9:7.1f} GB {num_fit:6.0f} {ridge:7.0f}   {verdict}')

# Exercise 4: plot the gap

In [ ]:
context_lens = np.array([512,1024,2048,4096,8192,16384,32768])

plt.figure(figsize=(7.5,4.5))
for name,(bandwidth, flops, vram) in cards.items():
  kv_room = vram - weight_bytes
  plt.plot(context_lens, kv_room/(context_lens*per_token), 'o-', label=name)
  plt.axhline(flops/bandwidth, ls=':', alpha=.5)

plt.xscale('log', base=2)
plt.yscale('log')
plt.xlabel('Context length')
plt.ylabel('Concurrent sequences that fit')
plt.title('Solid: what memory allows.  Dotted: what the ridge wants.')
plt.legend()
plt.grid(alpha=.3)
plt.show()

### What the plot shows

At long context every card sits below its own dotted line. No card can hold
the batch that its arithmetic units want.

Look at the H100. It has the most memory bandwidth **and** the highest ridge
point. Its tensor cores grew faster than its memory. Better hardware made the
gap wider, not narrower.

This is why the course teaches software. No future card removes the gap.